# Text classification with representation / embedding language models

In [1]:
from datasets import load_dataset

In [2]:
data = load_dataset("rotten_tomatoes")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [3]:
data["train"][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

In [22]:
from transformers import pipeline

In [23]:
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [24]:
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="mps"
)
pipe

loading configuration file config.json from cache at /Users/aksonsamvarghee/.cache/huggingface/hub/models--cardiffnlp--twitter-roberta-base-sentiment-latest/snapshots/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "pos

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
loading configuration file config.json from cache at /Users/aksonsamvarghee/.cache/huggingface/hub/models--cardiffnlp--twitter-roberta-base-sentiment-latest/snapshots/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json
Model config RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_

TextClassificationPipeline: {'model': 'RobertaForSequenceClassification', 'dtype': 'float32', 'device': 'mps', 'input_modalities': 'text'}

In [25]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

In [37]:
y_pred = []

for output in tqdm(pipe(KeyDataset(data["test"], 'text')), total=len(data["test"])):
    if output["label"] == "positive":
        y_pred.append(1)
    else:
        y_pred.append(0)
    
assert len(data["test"]) == len(y_pred)

100%|█████████████████████████████████████████████████| 1066/1066 [00:23<00:00, 45.71it/s]


In [3]:
from sklearn.metrics import classification_report

In [4]:
def eval_performance(y_true, y_pred):
    return classification_report(
        y_true, y_pred,
    )

In [39]:
from sklearn.metrics import confusion_matrix

In [40]:
confusion_matrix(data["test"]["label"], y_pred)

array([[502,  31],
       [234, 299]])

In [6]:

from sentence_transformers import SentenceTransformer
import os

In [10]:
from huggingface_hub import login


login(HF_TOKEN)

In [13]:
# train a classifier with sentence embedding
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="mps")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

train_embeddings.shape, test_embeddings.shape

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

((8530, 768), (1066, 768))

In [15]:
from sklearn.linear_model import LogisticRegression

In [51]:
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])
y_pred = clf.predict(test_embeddings)
print(eval_performance(data["test"]["label"], y_pred))

              precision    recall  f1-score   support

           0       0.85      0.86      0.85       533
           1       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



# Text classification with generative language models

In [3]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [4]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small", device_map="mps")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
device = "mps"

In [6]:
input_text = f"""
Given the document, review them as positive or negative.

{data["train"][0, -1]["text"]}

If positive respond 1, if negative respond 0. Do not answer anything else.
"""

input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

outputs = model.generate(input_ids)


/Users/aksonsamvarghee/work/studies/ml_prax/hoLLM/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [9]:
tokenizer.decode(outputs[0])

'<pad> negative</s>'

In [11]:
data["train"][0, -1]["label"]

[1, 0]